<a href="https://colab.research.google.com/github/AdelineKwakye/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
# TODO
print("df shape: \n", df.shape)
print("print types: \n",df.dtypes)
print("Null count per column: \n",df.isnull().sum())
print("Number of duplicate rows: \n", df.duplicated().sum())

df shape: 
 (8, 6)
print types: 
 order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
Null count per column: 
 order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
Number of duplicate rows: 
 1


**What is wrong with this data?** List at least five specific problems:

1. There is 1 duplicate row
2. There is 1 missing item value
3. There is 1 missing quantity value
4. There is 1 missing timestamp
5. The price column is stored as an object rather than a float or numeric value

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

# TODO: log('duplicates', 'dropped exact duplicate rows', removed)
log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
# TODO: clean['price'] = ...
clean['price'] = (clean['price'].str.replace('$', '', regex = False).str.strip().astype(float))
assert clean['price'].dtype == float
# TODO: log(...) -- note that price arrived as text
log('price', 'stripped $ and whitespaces and converted text to be a float',len(clean))

[price] stripped $ and whitespaces and converted text to be a float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [6]:
# TODO: clean['qty'] = pd.to_numeric(...)
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities

# TODO: apply your decision, then log both separately
clean = clean.dropna(subset=['qty']).copy()
log('missing qty', 'dropped missing qty because we do not know how many items were sold', missing)
log('negative qty', 'kept because they represent a refund/return', negative)

[missing qty] dropped missing qty because we do not know how many items were sold (1 row(s))
[negative qty] kept because they represent a refund/return (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [7]:
print('before:', sorted(clean['category'].unique()))

before_category = clean['category'].nunique()
# TODO: lowercase, strip, remove punctuation

clean['category'] = (
    clean['category']
    .str.lower()
    .str.strip()
    .str.replace('-', '', regex=False)
)

# TODO: CATEGORY_MAP = {...} for the judgment calls

CATEGORY_MAP = {
    'food': 'food',
    'merch': 'merch',
    'apparel': 'apparel',
    'raingear': 'raingear'
}

# print('after: ', sorted(clean['category'].unique()))


print('after: ', sorted(clean['category'].unique()))

after_category = clean['category'].nunique()

log(
    'categories',
    'normalized case and punctuation and mapped category variants to explicit names',
    before_category - after_category
)

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['apparel', 'food', 'merch', 'raingear']
[categories] normalized case and punctuation and mapped category variants to explicit names (2 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [8]:
# TODO
print('before:', sorted(clean['item'].dropna().unique()))
before_category = clean['item'].nunique()

clean['item'] = clean['item'].str.lower().str.strip()

ITEM_MAP = {
    'cheese burger': 'cheeseburger'
}


missing_item = clean['item'].isna().sum()
clean = clean.dropna(subset=['item']).copy()

print('after:', sorted(clean['item'].unique()))
after_category = clean['item'].nunique()


log('item names', 'normalized item names and mapped cheese burger to cheeseburger', before_category - after_category)
log('missing items', 'dropped rows with missing item because the product cannot be identified', missing_item)

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after: ['cheese burger', 'cheeseburger', 'rain poncho', 'uva t-shirt']
[item names] normalized item names and mapped cheese burger to cheeseburger (1 row(s))
[missing items] dropped rows with missing item because the product cannot be identified (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [9]:
# TODO
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')
failed_ts = clean['ts'].isna().sum()
clean['hour'] = clean['ts'].dt.hour

log('timestamps', 'parsed timestamps into real datetimes and coerced invalid or missing timestamps to NaT', failed_ts)


[timestamps] parsed timestamps into real datetimes and coerced invalid or missing timestamps to NaT (3 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [10]:
# TODO: assertions
assert clean.duplicated().sum() == 0, 'duplicates remain'
assert clean['price'].dtype == float, 'price is not numeric'
assert clean['qty'].notna().all(), 'missing quantities remain'
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), 'ts is not a datetime'
assert clean['category'].str.islower().all(), 'inconsistent category text'
# TODO: clean['revenue'] = ...
clean['revenue'] = clean['qty'] * clean['price']

# TODO: print rows, units, revenue, distinct categories
print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows: 5
units: 6.0
revenue: 76.5
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [11]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,stripped $ and whitespaces and converted text ...,7
2,missing qty,dropped missing qty because we do not know how...,1
3,negative qty,kept because they represent a refund/return,1
4,categories,normalized case and punctuation and mapped cat...,2
5,item names,normalized item names and mapped cheese burger...,1
6,missing items,dropped rows with missing item because the pro...,1
7,timestamps,parsed timestamps into real datetimes and coer...,3


**The decision that mattered most:** keeping the negative quantity that represented a refund/ return
**Revenue with it:** 76.50
**Revenue without it:** 94.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
rows_after = 5            # TODO
revenue_after = 76.5         # TODO
biggest_decision = 'keeping the negative quantities which represented returns/refunds'    # TODO: which choice moved the number most
revenue_other_way = 94.5     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 76.5
decision that mattered: keeping the negative quantities which represented returns/refunds
revenue the other way: 94.5
